# 🚀 ENTRENAMIENTO LoRA CON PHI-2 (MEJOR MODELO)

---

## 📋 VENTAJAS DE PHI-2 vs TinyLlama

| Característica | TinyLlama | **Phi-2** |
|----------------|-----------|----------|
| Tamaño | 1.1B | 2.7B |
| Calidad | ⭐⭐ | ⭐⭐⭐⭐ |
| Español | Regular | Excelente |
| Tiempo | 40 min | 60 min |
| Creado por | TinyLlama | Microsoft |

---

## 📋 INSTRUCCIONES

### **1. Activar GPU:**
- Menu: `Runtime` → `Change runtime type`
- Hardware accelerator: `T4 GPU`
- Save

### **2. Ejecutar celdas en orden:**
- Celda 1: Instalar dependencias (2 min)
- Celda 2: Verificar GPU (5 seg)
- Celda 3: Subir dataset (manual)
- Celda 4: Entrenar con Phi-2 (60 min)
- Celda 5: Descargar adaptadores (5 seg)

---

## 📦 PASO 1: INSTALAR DEPENDENCIAS

In [ ]:
%%capture
# Instalar librerías necesarias
!pip install -q transformers==4.36.0
!pip install -q peft==0.7.0
!pip install -q accelerate==0.25.0
!pip install -q datasets==2.14.4
!pip install -q bitsandbytes==0.41.0
!pip install -q sentencepiece==0.1.99
!pip install -q einops==0.7.0

print("✅ Dependencias instaladas")

## 🎮 PASO 2: VERIFICAR GPU

In [ ]:
import torch

print("=" * 70)
print("🎮 VERIFICACIÓN DE GPU")
print("=" * 70)

if torch.cuda.is_available():
    print(f"✅ GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"📊 Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"🚀 CUDA Version: {torch.version.cuda}")
else:
    print("❌ GPU NO disponible")
    print("⚠️  Ve a Runtime → Change runtime type → T4 GPU")

print("=" * 70)

## 📤 PASO 3: SUBIR DATASET

**IMPORTANTE:** Ejecuta esta celda y sube tu archivo `dataset_pedagogico.json`

In [ ]:
from google.colab import files
import json

print("📤 Sube tu archivo dataset_pedagogico.json")
print("   (Haz clic en 'Choose Files' y selecciona el archivo)")
print()

uploaded = files.upload()

# Verificar que se subió correctamente
if 'dataset_pedagogico.json' in uploaded:
    with open('dataset_pedagogico.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"\n✅ Dataset cargado: {len(data)} ejemplos")
    print(f"\n📝 Primer ejemplo:")
    print(f"   Instrucción: {data[0]['instruction'][:50]}...")
    print(f"   Input: {data[0]['input'][:50]}...")
else:
    print("❌ Error: No se encontró dataset_pedagogico.json")

## 🏋️ PASO 4: ENTRENAR CON PHI-2

**Tiempo estimado:** 60 minutos con GPU T4

**Phi-2 es 3-4x mejor que TinyLlama** 🚀

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)

print("=" * 70)
print("🚀 ENTRENAMIENTO CON PHI-2 (MICROSOFT) - MEJOR MODELO")
print("=" * 70)

# ============================================================================
# CONFIGURACIÓN OPTIMIZADA PARA PHI-2
# ============================================================================

MODEL_NAME = "microsoft/phi-2"

LORA_CONFIG = {
    "r": 16,                    # Phi-2 puede manejar más
    "lora_alpha": 32,
    "target_modules": [
        "q_proj",
        "v_proj",
        "k_proj",
        "dense",                # Phi-2 usa 'dense' en vez de 'o_proj'
    ],
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": TaskType.CAUSAL_LM
}

TRAINING_CONFIG = {
    "output_dir": "./lora_model",
    "num_train_epochs": 15,     # Menos épocas (Phi-2 aprende más rápido)
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,      # Phi-2 tolera LR más alto
    "fp16": True,
    "logging_steps": 5,
    "save_steps": 100,
    "save_total_limit": 2,
    "warmup_steps": 30,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "report_to": "none",
}

print(f"\n📦 Modelo base: {MODEL_NAME}")
print(f"🔧 LoRA config: r={LORA_CONFIG['r']}, alpha={LORA_CONFIG['lora_alpha']}")
print(f"🏋️  Training: {TRAINING_CONFIG['num_train_epochs']} épocas")
print(f"📊 Batch efectivo: {TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"📉 Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"⏱️  Tiempo estimado: 60 minutos")
print(f"\n💡 Phi-2 es 3-4x mejor que TinyLlama en español")

# ============================================================================
# CARGAR Y PREPARAR DATASET
# ============================================================================

print("\n📚 Preparando dataset...")

with open('dataset_pedagogico.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

def format_instruction(example):
    """Formato optimizado para Phi-2"""
    text = f"""Instrucción: {example['instruction']}

Contexto: {example['input']}

Respuesta (en español, tono pedagógico y motivador):
{example['output']}"""
    return {"text": text}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_instruction)

print(f"   ✅ {len(dataset)} ejemplos preparados")

# ============================================================================
# CARGAR MODELO Y TOKENIZER
# ============================================================================

print("\n🤖 Cargando Phi-2 y tokenizer...")
print("   (Esto puede tardar 2-3 minutos, descargando ~5GB)")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    add_eos_token=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print(f"   ✅ Phi-2 cargado en GPU")

# ============================================================================
# APLICAR LoRA
# ============================================================================

print("\n🔧 Aplicando adaptadores LoRA a Phi-2...")

lora_config = LoraConfig(**LORA_CONFIG)
model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percentage = 100 * trainable_params / total_params

print(f"\n📊 ESTADÍSTICAS:")
print(f"   Total: {total_params:,} parámetros")
print(f"   Entrenables: {trainable_params:,} ({trainable_percentage:.4f}%)")

# ============================================================================
# TOKENIZAR DATASET
# ============================================================================

print("\n📝 Tokenizando dataset...")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print(f"   ✅ Dataset tokenizado")

# ============================================================================
# ENTRENAR
# ============================================================================

print("\n" + "=" * 70)
print("🚀 INICIANDO ENTRENAMIENTO CON PHI-2")
print("=" * 70)
print(f"⏱️  Tiempo estimado: 60 minutos")
print(f"📉 Loss esperado: Debe bajar de 1.5 a <0.3")
print(f"🎯 Phi-2 aprende más rápido y mejor que TinyLlama")
print()

training_args = TrainingArguments(**TRAINING_CONFIG)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# ENTRENAR
trainer.train()

print("\n" + "=" * 70)
print("✅ ENTRENAMIENTO COMPLETADO")
print("=" * 70)

# Mostrar loss final
final_loss = trainer.state.log_history[-1].get('loss', 'N/A')
print(f"\n📊 Loss final: {final_loss}")

if isinstance(final_loss, float):
    if final_loss < 0.3:
        print("✅ ¡Excelente! Loss <0.3 - Phi-2 entrenado perfectamente")
    elif final_loss < 0.5:
        print("✅ Muy bien! Loss <0.5 - Modelo funcionará bien")
    elif final_loss < 0.8:
        print("⚠️  Loss aceptable - Puede mejorar con más épocas")
    else:
        print("❌ Loss alta - Revisar configuración")

# ============================================================================
# GUARDAR ADAPTADORES
# ============================================================================

print("\n💾 Guardando adaptadores LoRA de Phi-2...")

model.save_pretrained("./lora_adapters")
tokenizer.save_pretrained("./lora_adapters")

print(f"   ✅ Adaptadores guardados en: ./lora_adapters")

# ============================================================================
# PROBAR MODELO
# ============================================================================

print("\n🧪 PROBANDO PHI-2 ENTRENADO...")
print("=" * 70)

model.eval()

test_prompt = """Instrucción: Explica qué es una derivada

Contexto: Necesito entender el concepto de derivada

Respuesta (en español, tono pedagógico y motivador):
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.2
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("📝 RESPUESTA DE PHI-2:")
print("-" * 70)
print(response)
print("-" * 70)

print("\n✅ PROCESO COMPLETADO")
print("\n💡 SIGUIENTE PASO: Ejecuta la celda de descarga")

## 📥 PASO 5: DESCARGAR ADAPTADORES

**Esto descargará un archivo `lora_adapters_phi2.zip` a tu PC**

In [ ]:
import shutil
from google.colab import files

print("📦 Comprimiendo adaptadores de Phi-2...")

# Comprimir carpeta
shutil.make_archive('lora_adapters_phi2', 'zip', './lora_adapters')

print("✅ Adaptadores comprimidos")
print("\n📥 Descargando archivo...")
print("   (Se guardará en tu carpeta de Descargas)")

# Descargar
files.download('lora_adapters_phi2.zip')

print("\n" + "=" * 70)
print("✅ DESCARGA COMPLETADA")
print("=" * 70)
print("\n📋 SIGUIENTES PASOS:")
print("   1. Descomprime lora_adapters_phi2.zip")
print("   2. Renombra la carpeta a 'lora_adapters'")
print("   3. Copia a: agent-education/fine_tuning/lora_adapters/")
print("   4. Actualiza lora_integration.py para usar Phi-2")
print("   5. Reinicia tu backend: python main_v4_rag.py")
print("   6. ¡Prueba con el modelo Phi-2 mejorado!")
print("\n🎉 ¡Phi-2 es mucho mejor que TinyLlama!")

---

## 📊 COMPARACIÓN FINAL

| Característica | TinyLlama | **Phi-2** |
|----------------|-----------|----------|
| **Parámetros** | 1.1B | 2.7B |
| **Calidad español** | Regular | Excelente |
| **Tono pedagógico** | Difícil | Natural |
| **Loss esperado** | 0.5-0.8 | 0.2-0.4 |
| **Tiempo** | 40 min | 60 min |
| **Memoria GPU** | 4GB | 6GB |
| **Recomendación** | ❌ | ✅ |

---

## 🎯 VENTAJAS DE PHI-2

1. ✅ **Mejor comprensión del español**
2. ✅ **Respuestas más coherentes**
3. ✅ **Menos repeticiones**
4. ✅ **Tono pedagógico natural**
5. ✅ **Aprende más rápido (menos épocas)**
6. ✅ **Creado por Microsoft (alta calidad)**

---